In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IntrusionDetection") \
    .master("local[2]") \
    .getOrCreate()

df = spark.read.parquet("hdfs://namenode:8020/data/processed/features.parquet")
print(df.count())

2730540


In [2]:
from pyspark.sql.functions import row_number, from_unixtime, lit, to_timestamp
from pyspark.sql.window import Window

w = Window.orderBy(lit(1))
df = df.withColumn(
    "timestamp",
    to_timestamp(from_unixtime(lit(1421927414) + row_number().over(w)))
)

In [3]:
from pyspark.sql.functions import window, count, when, col

df.filter(col("attack_cat").isNotNull())\
  .groupBy(window("timestamp", "5 minutes"), "attack_cat")\
  .count()\
  .orderBy("window")\
  .show(10)

+--------------------+--------------+-----+
|              window|    attack_cat|count|
+--------------------+--------------+-----+
|{2015-01-26 23:20...|      Exploits|    7|
|{2015-01-26 23:20...|Reconnaissance|    2|
|{2015-01-26 23:20...|      Backdoor|    1|
|{2015-01-26 23:20...|           DoS|    7|
|{2015-01-26 23:20...|       Generic|    1|
|{2015-01-26 23:20...|       Fuzzers|    1|
|{2015-01-26 23:25...|      Exploits|   24|
|{2015-01-26 23:25...|       Fuzzers|    6|
|{2015-01-26 23:25...|           DoS|   15|
|{2015-01-26 23:25...|Reconnaissance|    5|
+--------------------+--------------+-----+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import hour

df.filter(col("attack_cat").isNotNull())\
  .groupBy(hour("timestamp").alias("hour"))\
  .pivot("attack_cat")\
  .count()\
  .orderBy("hour")\
  .show()

+----+--------+--------+----+--------+-------+-------+--------------+---------+-----+
|hour|Analysis|Backdoor| DoS|Exploits|Fuzzers|Generic|Reconnaissance|Shellcode|Worms|
+----+--------+--------+----+--------+-------+-------+--------------+---------+-----+
|   0|     481|     505|2111|    3614|   1551|  10572|           696|       72|    6|
|   1|     223|     235|1246|    2869|   1374|   9512|           635|       62|   12|
|   2|     108|     122| 872|    2254|   1290|  10716|           622|       71|   10|
|   3|      24|      51| 356|    1519|   1484|  10640|           540|       70|    8|
|   4|      27|      29| 243|    1255|   1222|  10338|           508|       68|    6|
|   5|      10|      30| 227|    1245|   1046|  10115|           447|       52|    4|
|   6|      25|      27| 217|    1215|    724|   8832|           466|       51|    6|
|   7|      46|      36| 224|    1294|    855|   8472|           528|       58|    3|
|   8|     121|      42| 317|    1484|   1048|   7512|

In [5]:
from pyspark.sql.functions import avg, max, min, stddev

df.filter(col("attack_cat").isNotNull())\
  .groupBy("attack_cat")\
  .agg(
      avg("threat_score").alias("avg_threat"),
      max("threat_score").alias("max_threat"),
      stddev("threat_score").alias("threat_volatility")
  )\
  .orderBy("avg_threat", ascending=False)\
  .show()

+--------------+------------------+------------------+------------------+
|    attack_cat|        avg_threat|        max_threat| threat_volatility|
+--------------+------------------+------------------+------------------+
|      Exploits|2.6862615096596656|1002.3521230103261|25.729815668820052|
|       Fuzzers| 2.090706344947893|150.66483417634745| 4.566521386114914|
|           DoS|2.0089341285554934| 811.9810516047745|22.892024346954585|
|         Worms| 1.354295927611828|12.501940610658572|1.9995838436284052|
|     Shellcode|1.1641816155027316| 18.87041786734175|1.3901262943080603|
|Reconnaissance| 0.759216370199122|48.163934534291855|0.8362847325581221|
|       Generic|0.6121183824705186|250.50160108460193|1.5630097488969208|
|      Backdoor|0.5562468963489904|48.163934534291855|1.6288423324031513|
|      Analysis| 0.532823995466067|48.163934534291855|1.5308175266487098|
+--------------+------------------+------------------+------------------+

